# Chains and anchored archives

Install Deno and register its TypeScript kernel with `deno jupyter --install`.
Open this file in a Jupyter front end such as JupyterLab or VS Code, select
**Deno**, then Run All in a fresh kernel. The cells import the exact published
verifier **2.0.0**.

Verification returns facts; the host still owns trusted inputs, authorization,
replay and revocation. Only safe semantic results are displayed; generated keys,
signatures and nonces stay in memory.

## Load the published verifier

Check the package identity and the portable backend Deno selects.


In [ ]:
import { Buffer } from "node:buffer";
import {
  assembleCompact,
  BACKEND_ID,
  boundaryAnchorSigningInput,
  encodeAnchoredExport,
  encodeConsumptionEntry,
  type HistoricalPublicKey,
  keyTransitionSigningInput,
  publicKeyThumbprintRaw,
  type Result,
  type SigningInput,
  verifyAnchoredExport,
  verifyHistoricalAnchor,
} from "npm:@bounded-authority-protocol/verifier@2.0.0";
import {
  createHash,
  generateKeyPairSync,
  type KeyObject,
  sign,
} from "node:crypto";
import verifierPackage from "npm:@bounded-authority-protocol/verifier@2.0.0/package.json" with {
  type: "json",
};
import assert from "node:assert/strict";
assert.equal(verifierPackage.version, "2.0.0");
assert.equal(BACKEND_ID, "portable");
console.log(
  JSON.stringify({ version: verifierPackage.version, backend: BACKEND_ID }),
);

{"version":"2.0.0","backend":"portable"}


## Prepare compact signing

Assemble exactly the standard JWS protected and payload segments before signing.


In [ ]:
const publicKeyBytes = (key: KeyObject): Uint8Array =>
  Buffer.from(key.export({ format: "jwk" }).x!, "base64url");
function signCompact(
  input: Result<SigningInput>,
  privateKey: KeyObject,
): Uint8Array {
  assert.ok(input.ok);
  const signingBytes = Buffer.concat([
    input.value.protectedSegment,
    Buffer.from("."),
    input.value.payloadSegment,
  ]);
  const compact = assembleCompact(
    input.value,
    sign(null, signingBytes, privateKey),
  );
  assert.ok(compact.ok);
  return compact.value;
}

console.log(JSON.stringify({ signer: typeof signCompact }));

{"signer":"function"}


## Build consumption rows

Each row commits to the previous hash and its own commitment.


In [ ]:
// Three rows.
const chainId = "audit:approvals";
const zero = new Uint8Array(32);
const rows: Uint8Array[] = [];
let head: Uint8Array = zero;
for (let sequence = 1; sequence <= 3; sequence++) {
  const commitment = createHash("sha256").update(`approval ${sequence}`)
    .digest();
  const entry = encodeConsumptionEntry({
    chainId,
    sequence,
    previousHash: head,
    commitment,
  });
  assert.ok(entry.ok);
  rows.push(entry.value.bytes);
  head = entry.value.hash;
}

console.log(JSON.stringify({ rowCount: rows.length, headBytes: head.length }));

{"rowCount":3,"headBytes":32}


## Enroll historical keys

Supply trusted historical public keys and validity windows explicitly.


In [ ]:
// Two archive keys with overlapping windows.
const currentArchiveKeyPair = generateKeyPairSync("ed25519");
const nextArchiveKeyPair = generateKeyPairSync("ed25519");
const keyA: HistoricalPublicKey = {
  keyId: "archive-a",
  publicKey: publicKeyBytes(currentArchiveKeyPair.publicKey),
  validFrom: 1_000,
  validBefore: 3_000,
};
const keyB: HistoricalPublicKey = {
  keyId: "archive-b",
  publicKey: publicKeyBytes(nextArchiveKeyPair.publicKey),
  validFrom: 2_000,
  validBefore: null,
};
const fingerprintA = publicKeyThumbprintRaw(keyA.publicKey);
const fingerprintB = publicKeyThumbprintRaw(keyB.publicKey);
assert.equal(keyA.publicKey.length, 32);
assert.equal(keyB.publicKey.length, 32);
console.log(
  JSON.stringify({
    currentKeyBytes: keyA.publicKey.length,
    nextKeyBytes: keyB.publicKey.length,
  }),
);

{"currentKeyBytes":32,"nextKeyBytes":32}


## Sign the start boundary

The current key signs the genesis boundary.


In [ ]:
const startAnchor = signCompact(
  boundaryAnchorSigningInput({
    anchorId: "anchor-0",
    anchoredAt: 1_500,
    chainId,
    sequence: 0,
    chainHash: zero,
    keyId: "archive-a",
    publicKey: keyA.publicKey,
  }),
  currentArchiveKeyPair.privateKey,
);
assert.equal(new TextDecoder().decode(startAnchor).split(".").length, 3);
console.log(
  JSON.stringify({
    compactSegments: new TextDecoder().decode(startAnchor).split(".").length,
  }),
);

{"compactSegments":3}


## Sign the key handover

The current key names the next key and the handover time.


In [ ]:
const transition = signCompact(
  keyTransitionSigningInput({
    transitionId: "rotation-1",
    chainId,
    effectiveAt: 2_500,
    currentKeyId: "archive-a",
    currentPublicKey: keyA.publicKey,
    nextKeyId: "archive-b",
    nextPublicKey: keyB.publicKey,
  }),
  currentArchiveKeyPair.privateKey,
);
assert.equal(new TextDecoder().decode(transition).split(".").length, 3);
console.log(
  JSON.stringify({
    compactSegments: new TextDecoder().decode(transition).split(".").length,
  }),
);

{"compactSegments":3}


## Sign the end boundary

The next key signs the final chain boundary.


In [ ]:
const endAnchor = signCompact(
  boundaryAnchorSigningInput({
    anchorId: "anchor-3",
    anchoredAt: 2_600,
    chainId,
    sequence: 3,
    chainHash: head,
    keyId: "archive-b",
    publicKey: keyB.publicKey,
  }),
  nextArchiveKeyPair.privateKey,
);
assert.equal(new TextDecoder().decode(endAnchor).split(".").length, 3);
console.log(
  JSON.stringify({
    compactSegments: new TextDecoder().decode(endAnchor).split(".").length,
  }),
);

{"compactSegments":3}


## Verify the expected boundary

Expected identifiers, fingerprints and chain bounds come from the caller.


In [ ]:
// What the verifier expects, from its own records.
const chain = {
  chainId,
  firstSequence: 1,
  lastSequence: 3,
  rowCount: 3,
  previousHash: zero,
  lastHash: head,
};
const expectedStart = {
  anchorId: "anchor-0",
  anchoredAt: 1_500,
  chainId,
  sequence: 0,
  chainHash: zero,
  keyId: "archive-a",
  keyFingerprint: fingerprintA,
};
const expectedEnd = {
  anchorId: "anchor-3",
  anchoredAt: 2_600,
  chainId,
  sequence: 3,
  chainHash: head,
  keyId: "archive-b",
  keyFingerprint: fingerprintB,
};
const expectedTransition = {
  transitionId: "rotation-1",
  chainId,
  effectiveAt: 2_500,
  currentKeyId: "archive-a",
  currentKeyFingerprint: fingerprintA,
  nextKeyId: "archive-b",
  nextKeyFingerprint: fingerprintB,
};

// A single anchor verifies on its own against its key.
const anchorFacts = verifyHistoricalAnchor(startAnchor, keyA, expectedStart);
assert.ok(anchorFacts.ok);
assert.equal(anchorFacts.value.verification, "signature_and_window");

console.log(
  JSON.stringify({
    ok: anchorFacts.ok,
    verification: anchorFacts.value.verification,
  }),
);

{"ok":true,"verification":"signature_and_window"}


## Encode the archive

Encoding preserves canonical row and boundary bytes.


In [ ]:
// Encode. In major 1 the export context also has digest and objectVersion members, which encoding
// does not read (the v2 and v3 namespaces split the two contexts); any placeholder will do here.
const encoded = encodeAnchoredExport(
  { rows, startAnchor, endAnchor, transitions: [transition], ...chain },
  {
    chain,
    digest: zero,
    objectVersion: "unused",
    startAnchor: expectedStart,
    endAnchor: expectedEnd,
    transitions: [expectedTransition],
  },
);
assert.ok(encoded.ok);

console.log(
  JSON.stringify({ ok: encoded.ok, digestBytes: encoded.value.digest.length }),
);

{"ok":true,"digestBytes":32}


## Verify the stored object

Supply the storage version and digest as expected context.


In [ ]:
// Store it (here: split into two chunks), then verify the stored object.
const archive = encoded.value.archive;
const stored = {
  chunks: [archive.subarray(0, 100), archive.subarray(100)],
  version: "object-version-7",
};
const exported = verifyAnchoredExport(stored, { keys: [keyA, keyB] }, {
  chain,
  digest: encoded.value.digest,
  objectVersion: "object-version-7",
  startAnchor: expectedStart,
  endAnchor: expectedEnd,
  transitions: [expectedTransition],
});
assert.ok(exported.ok);
assert.equal(exported.value.rowCount, 3);
assert.equal(exported.value.transitionCount, 1);
assert.equal(exported.value.verification, "anchored_export");

console.log(
  JSON.stringify({
    ok: exported.ok,
    rowCount: exported.value.rowCount,
    transitionCount: exported.value.transitionCount,
    authorization: exported.value.authorization,
  }),
);

{"ok":true,"rowCount":3,"transitionCount":1,"authorization":"not_evaluated"}


## Refuse changed bytes

Flipping one byte must invalidate the stored object digest.


In [ ]:
const tampered = Buffer.from(archive);
tampered[tampered.length - 1] = tampered[tampered.length - 1]! ^ 1;
const tamperedResult = verifyAnchoredExport(
  { chunks: [tampered], version: "object-version-7" },
  { keys: [keyA, keyB] },
  {
    chain,
    digest: encoded.value.digest,
    objectVersion: "object-version-7",
    startAnchor: expectedStart,
    endAnchor: expectedEnd,
    transitions: [expectedTransition],
  },
);
assert.deepEqual(tamperedResult, { ok: false });
console.log(JSON.stringify(tamperedResult));

{"ok":false}


## Finish

No private material was written to disk. Shut down the kernel to discard the
demonstration keys. Continue with the
[getting started guide](../guides/getting-started.md).